In [ ]:
# Image Transformation Pipeline with PyTorch
# Part 3: Complete Pipeline (35 minutes)

"""
This notebook covers building complete transformation pipelines.
Topics covered:
- Building training and validation pipelines
- Best practices for data augmentation
- Working with real datasets (CIFAR-10)
- Creating custom transformations
- Integration with DataLoader
- Visualization and debugging
- Performance optimization
"""

# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
from typing import Tuple, List

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ============================================================================
# SECTION 1: BUILDING TRAINING AND VALIDATION PIPELINES (15 minutes)
# ============================================================================

print("\n" + "="*80)
print("SECTION 1: BUILDING TRAINING AND VALIDATION PIPELINES")
print("="*80)

"""
Key principles for building transformation pipelines:

TRAINING PIPELINE:
1. Geometric augmentation (RandomResizedCrop, Flip, Rotation)
2. Color augmentation (ColorJitter)
3. Advanced augmentation (Blur, Erasing)
4. Normalization
Goal: Increase diversity, prevent overfitting

VALIDATION PIPELINE:
1. Deterministic resizing
2. Center crop
3. Normalization (same as training!)
Goal: Consistent, reproducible evaluation
"""

# ============================================================================
# 1.1 STANDARD IMAGENET-STYLE PIPELINE
# ============================================================================

print("\n1.1 STANDARD IMAGENET-STYLE PIPELINE")
print("-" * 80)

"""
This is the standard pipeline used for ImageNet and transfer learning.
"""

# ImageNet statistics (used for normalization)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms
train_transforms_imagenet = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.33), ratio=(0.3, 3.3))
])

# Validation transforms
val_transforms_imagenet = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Training Pipeline:")
print("  1. RandomResizedCrop(224) - scale=(0.08, 1.0)")
print("  2. RandomHorizontalFlip(p=0.5)")
print("  3. ColorJitter - moderate strength")
print("  4. ToTensor")
print("  5. Normalize with ImageNet stats")
print("  6. RandomErasing(p=0.25)")

print("\nValidation Pipeline:")
print("  1. Resize(256)")
print("  2. CenterCrop(224)")
print("  3. ToTensor")
print("  4. Normalize with ImageNet stats")

# ============================================================================
# 1.2 CIFAR-10 STYLE PIPELINE (SMALLER IMAGES)
# ============================================================================

print("\n1.2 CIFAR-10 STYLE PIPELINE (32x32 images)")
print("-" * 80)

"""
For smaller images like CIFAR-10 (32x32), we need different augmentation:
- No resize (images are already small)
- Padding + RandomCrop instead of RandomResizedCrop
- Smaller rotation angles
- More conservative augmentation
"""

# CIFAR-10 statistics
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD = [0.2470, 0.2435, 0.2616]

# Training transforms for CIFAR-10
train_transforms_cifar = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # Pad then crop
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
])

# Validation transforms for CIFAR-10
val_transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

print("Training Pipeline (CIFAR-10):")
print("  1. RandomCrop(32, padding=4)")
print("  2. RandomHorizontalFlip(p=0.5)")
print("  3. ColorJitter - conservative")
print("  4. ToTensor")
print("  5. Normalize with CIFAR-10 stats")
print("  6. RandomErasing(p=0.2)")

print("\nValidation Pipeline (CIFAR-10):")
print("  1. ToTensor")
print("  2. Normalize with CIFAR-10 stats")
print("  (No geometric transforms needed - already 32x32)")

# ============================================================================
# 1.3 LOADING CIFAR-10 DATASET
# ============================================================================

print("\n1.3 LOADING CIFAR-10 DATASET")
print("-" * 80)

# Download and load CIFAR-10
print("Downloading CIFAR-10 dataset...")
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transforms_cifar
)

val_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=val_transforms_cifar
)

print(f"\nTraining set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes}")

# ============================================================================
# 1.4 VISUALIZING AUGMENTED SAMPLES
# ============================================================================

print("\n1.4 VISUALIZING AUGMENTED SAMPLES")
print("-" * 80)

def denormalize(tensor, mean, std):
    """Denormalize a tensor for visualization"""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(tensor, 0, 1)

def show_augmented_samples(dataset, num_samples=8, same_image=True):
    """Show multiple augmented versions of the same image"""
    # Get one image
    if same_image:
        # Get original image without transforms
        original_dataset = torchvision.datasets.CIFAR10(
            root='./data',
            train=True,
            download=False,
            transform=None
        )
        idx = np.random.randint(0, len(original_dataset))
        original_img, label = original_dataset[idx]
        
        # Apply transforms multiple times
        images = []
        for _ in range(num_samples):
            img_tensor, _ = dataset.dataset.transform(original_img), label
            img_denorm = denormalize(img_tensor, CIFAR10_MEAN, CIFAR10_STD)
            images.append(img_denorm)
        
        # Add original
        original_tensor = transforms.ToTensor()(original_img)
        images = [original_tensor] + images
        
    else:
        # Get different images
        indices = np.random.choice(len(dataset), num_samples, replace=False)
        images = []
        for idx in indices:
            img, _ = dataset[idx]
            img_denorm = denormalize(img, CIFAR10_MEAN, CIFAR10_STD)
            images.append(img_denorm)
    
    # Plot
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(2*n, 2))
    if n == 1:
        axes = [axes]
    
    for idx, (img, ax) in enumerate(zip(images, axes)):
        img_np = img.permute(1, 2, 0).numpy()
        ax.imshow(img_np)
        if idx == 0 and same_image:
            ax.set_title("Original")
        else:
            ax.set_title(f"Aug {idx}" if same_image else f"Sample {idx+1}")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Show augmented versions of the same image
print("\nShowing multiple augmentations of the SAME image:")
show_augmented_samples(train_dataset, num_samples=8, same_image=True)

# Show different images from validation set
print("\nShowing different images from VALIDATION set (no augmentation):")
show_augmented_samples(val_dataset, num_samples=8, same_image=False)

# ============================================================================
# SECTION 2: CUSTOM TRANSFORMATIONS (10 minutes)
# ============================================================================

print("\n" + "="*80)
print("SECTION 2: CUSTOM TRANSFORMATIONS")
print("="*80)

"""
Creating custom transformations allows you to:
- Implement domain-specific augmentations
- Combine multiple operations
- Add conditional logic
"""

# ============================================================================
# 2.1 SIMPLE CUSTOM TRANSFORMATION
# ============================================================================

print("\n2.1 SIMPLE CUSTOM TRANSFORMATION")
print("-" * 80)

class AddGaussianNoise:
    """Add Gaussian noise to image tensor"""
    
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std
    
    def __call__(self, tensor):
        """
        Args:
            tensor (Tensor): Image tensor of shape (C, H, W)
        Returns:
            Tensor: Noisy image tensor
        """
        noise = torch.randn_like(tensor) * self.std + self.mean
        noisy_tensor = tensor + noise
        return torch.clamp(noisy_tensor, 0, 1)
    
    def __repr__(self):
        return f"{self.__class__.__name__}(mean={self.mean}, std={self.std})"

# Example usage
custom_transform = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(mean=0.0, std=0.05),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

print("Custom transformation: AddGaussianNoise")
print(custom_transform)

# ============================================================================
# 2.2 ADVANCED CUSTOM TRANSFORMATION
# ============================================================================

print("\n2.2 ADVANCED CUSTOM TRANSFORMATION")
print("-" * 80)

class Cutout:
    """Randomly mask out square regions of the image"""
    
    def __init__(self, n_holes=1, length=16):
        """
        Args:
            n_holes (int): Number of patches to cut out
            length (int): Length of the square patches
        """
        self.n_holes = n_holes
        self.length = length
    
    def __call__(self, img):
        """
        Args:
            img (Tensor): Image tensor of shape (C, H, W)
        Returns:
            Tensor: Image with cutout applied
        """
        h = img.size(1)
        w = img.size(2)
        
        mask = torch.ones((h, w), dtype=torch.float32)
        
        for _ in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)
            
            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)
            
            mask[y1:y2, x1:x2] = 0.0
        
        mask = mask.expand_as(img)
        img = img * mask
        
        return img
    
    def __repr__(self):
        return f"{self.__class__.__name__}(n_holes={self.n_holes}, length={self.length})"

# Example with custom Cutout
cutout_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    Cutout(n_holes=1, length=16)
])

print("Custom transformation: Cutout")
print(cutout_transform)

# Visualize Cutout
print("\nVisualizing Cutout transformation:")
test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=None
)

# Get one sample
img, label = test_dataset[42]

# Apply different versions
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

# Original
axes[0].imshow(img)
axes[0].set_title("Original")
axes[0].axis('off')

# Apply cutout multiple times
for i in range(1, 5):
    img_tensor = cutout_transform(img)
    img_denorm = denormalize(img_tensor, CIFAR10_MEAN, CIFAR10_STD)
    axes[i].imshow(img_denorm.permute(1, 2, 0))
    axes[i].set_title(f"Cutout {i}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# ============================================================================
# 2.3 CONDITIONAL CUSTOM TRANSFORMATION
# ============================================================================

print("\n2.3 CONDITIONAL CUSTOM TRANSFORMATION")
print("-" * 80)

class RandomTransformChoice:
    """Randomly choose one transformation from a list"""
    
    def __init__(self, transforms_list):
        """
        Args:
            transforms_list: List of transformation callables
        """
        self.transforms = transforms_list
    
    def __call__(self, img):
        """Randomly select and apply one transformation"""
        transform = np.random.choice(self.transforms)
        return transform(img)
    
    def __repr__(self):
        format_string = self.__class__.__name__ + '('
        for t in self.transforms:
            format_string += '\n    {0}'.format(t)
        format_string += '\n)'
        return format_string

# Example usage
random_choice_transform = RandomTransformChoice([
    transforms.ColorJitter(brightness=0.5),
    transforms.RandomRotation(15),
    transforms.GaussianBlur(kernel_size=5),
])

print("Random choice transformation:")
print(random_choice_transform)

# ============================================================================
# SECTION 3: INTEGRATION WITH DATALOADER (10 minutes)
# ============================================================================

print("\n" + "="*80)
print("SECTION 3: INTEGRATION WITH DATALOADER")
print("="*80)

# ============================================================================
# 3.1 CREATING DATALOADERS
# ============================================================================

print("\n3.1 CREATING DATALOADERS")
print("-" * 80)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Batch size: {batch_size}")

# ============================================================================
# 3.2 VISUALIZING BATCHES
# ============================================================================

print("\n3.2 VISUALIZING BATCHES")
print("-" * 80)

def show_batch(loader, dataset_name="Dataset", num_images=16):
    """Visualize a batch of images"""
    # Get one batch
    images, labels = next(iter(loader))
    
    # Denormalize
    images_denorm = torch.stack([
        denormalize(img, CIFAR10_MEAN, CIFAR10_STD)
        for img in images[:num_images]
    ])
    
    # Create grid
    grid = torchvision.utils.make_grid(images_denorm, nrow=4, padding=2)
    
    # Plot
    plt.figure(figsize=(12, 12))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"{dataset_name} Batch (showing {num_images}/{len(images)} images)")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Show labels
    class_names = train_dataset.classes
    print(f"\nLabels for first {num_images} images:")
    print([class_names[label] for label in labels[:num_images].tolist()])

print("\nTraining batch (with augmentation):")
show_batch(train_loader, "Training", num_images=16)

print("\nValidation batch (no augmentation):")
show_batch(val_loader, "Validation", num_images=16)

# ============================================================================
# 3.3 PERFORMANCE CONSIDERATIONS
# ============================================================================

print("\n3.3 PERFORMANCE CONSIDERATIONS")
print("-" * 80)

"""
Tips for optimizing data loading:
1. Use num_workers > 0 for parallel loading
2. Use pin_memory=True when using GPU
3. Keep transforms lightweight
4. Consider caching preprocessed data for validation
5. Use persistent_workers=True for long training
"""

# Benchmark data loading speed
def benchmark_dataloader(loader, num_batches=50):
    """Measure data loading speed"""
    start_time = time.time()
    for i, (images, labels) in enumerate(loader):
        if i >= num_batches:
            break
        # Simulate processing time
        pass
    end_time = time.time()
    
    elapsed = end_time - start_time
    images_per_sec = (num_batches * loader.batch_size) / elapsed
    
    return elapsed, images_per_sec

print("\nBenchmarking data loading speed...")
train_time, train_speed = benchmark_dataloader(train_loader, num_batches=50)
print(f"Training loader: {train_time:.2f}s for 50 batches")
print(f"Speed: {train_speed:.0f} images/second")

val_time, val_speed = benchmark_dataloader(val_loader, num_batches=50)
print(f"Validation loader: {val_time:.2f}s for 50 batches")
print(f"Speed: {val_speed:.0f} images/second")

# ============================================================================
# SECTION 4: BEST PRACTICES AND GUIDELINES
# ============================================================================

print("\n" + "="*80)
print("SECTION 4: BEST PRACTICES AND GUIDELINES")
print("="*80)

best_practices = """
DATA AUGMENTATION BEST PRACTICES:

1. START CONSERVATIVE, INCREASE GRADUALLY
   - Begin with minimal augmentation
   - Increase if you see overfitting (train >> val accuracy)
   - Monitor validation performance

2. MATCH AUGMENTATION TO YOUR PROBLEM
   - Object classification: RandomResizedCrop, Flip, Color
   - Fine-grained classification: Less aggressive crop
   - Medical imaging: Be very careful with color changes
   - Satellite imagery: Consider vertical flips, rotations

3. NORMALIZATION IS CRITICAL
   - Always normalize inputs
   - Use ImageNet stats for transfer learning
   - Compute dataset-specific stats for training from scratch
   - Same normalization for train and validation!

4. COMMON AUGMENTATION STRENGTHS:
   Small datasets (<10K images):
   - Strong augmentation needed
   - ColorJitter(0.4, 0.4, 0.4, 0.1)
   - RandomErasing(p=0.3)
   - Consider mixup/cutmix
   
   Medium datasets (10K-100K):
   - Moderate augmentation
   - ColorJitter(0.3, 0.3, 0.3, 0.05)
   - RandomErasing(p=0.2)
   
   Large datasets (>100K):
   - Conservative augmentation
   - ColorJitter(0.2, 0.2, 0.2, 0.05)
   - RandomErasing(p=0.1)

5. VALIDATION SET RULES:
   - NEVER augment validation/test data
   - Use deterministic transforms only
   - Same normalization as training
   - Consistent preprocessing

6. DEBUGGING AUGMENTATION:
   - Always visualize augmented images
   - Check if augmentation is too aggressive
   - Ensure labels still make sense after augmentation
   - Monitor training dynamics

7. PERFORMANCE OPTIMIZATION:
   - Use multiple workers in DataLoader
   - Use pin_memory for GPU training
   - Consider caching for validation sets
   - Profile to find bottlenecks

8. ADVANCED TECHNIQUES:
   - AutoAugment: Learned augmentation policies
   - RandAugment: Random augmentation with magnitude
   - MixUp: Mix two images and labels
   - CutMix: Replace regions between images
"""

print(best_practices)

# ============================================================================
# PRACTICAL EXAMPLE: TRAINING PIPELINE
# ============================================================================

print("\n" + "="*80)
print("PRACTICAL EXAMPLE: COMPLETE TRAINING PIPELINE")
print("="*80)

# Define a simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Initialize model
model = SimpleCNN(num_classes=10)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"\nModel initialized on device: {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training function (simplified)
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx >= 10:  # Only train on a few batches for demo
            break
    
    epoch_loss = running_loss / (batch_idx + 1)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

print("\nTraining for one epoch (demo - only 10 batches)...")
train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
print(f"Training Loss: {train_loss:.4f} | Training Acc: {train_acc:.2f}%")

print("\nThis demonstrates the complete pipeline:")
print("  1. Define transformations (train + validation)")
print("  2. Create datasets with transforms")
print("  3. Create DataLoaders")
print("  4. Train model with augmented data")
print("  5. Validate on clean data")

# ============================================================================
# KEY TAKEAWAYS - PART 3
# ============================================================================

print("\n" + "="*80)
print("KEY TAKEAWAYS - PART 3")
print("="*80)

takeaways = """
1. PIPELINE DESIGN:
   - Training: Augmentation + Normalization
   - Validation: Normalization only
   - Use appropriate transforms for image size

2. CUSTOM TRANSFORMATIONS:
   - Inherit or create callable classes
   - Implement __call__ and __repr__
   - Can add domain-specific augmentation

3. DATALOADER INTEGRATION:
   - Transforms applied during data loading
   - Use num_workers for parallel loading
   - Use pin_memory for GPU training
   - Always visualize batches!

4. BEST PRACTICES:
   - Start conservative, increase if overfitting
   - Match augmentation to problem domain
   - Always normalize (same stats for train/val)
   - Never augment validation data
   - Monitor training dynamics

5. DEBUGGING:
   - Visualize augmented images frequently
   - Check augmentation isn't too aggressive
   - Ensure labels still match after transforms
   - Benchmark data loading speed

6. ADVANCED:
   - AutoAugment/RandAugment for SOTA results
   - Mixup/CutMix for better generalization
   - Consider learned augmentation policies
   - Experiment with augmentation strength

REMEMBER:
Good augmentation = Better generalization = Higher validation accuracy!
"""

print(takeaways)

print("\n" + "="*80)
print("CONGRATULATIONS!")
print("You've completed the Image Transformation Pipeline course!")
print("="*80)